In [ ]:
import os
import glob
import time

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.colors import ListedColormap

import scipy
from scipy.optimize import curve_fit
from scipy import stats
from scipy import linalg
from scipy.stats import norm
from scipy.spatial import distance
from scipy.cluster import hierarchy

import pandas as pd

import sklearn
from sklearn import mixture

import itertools
import seaborn as sns

In [ ]:
master_dir = '/path/to/your/analysis/'
dirs = os.listdir(master_dir)
print(dirs)

output_dir = '/path/to/your/output'

In [ ]:
custom_colors = ['white', '#2ca02c', '#d62728']
custom_cmap = ListedColormap(custom_colors)

In [ ]:
def solve(m1,m2,std1,std2,s1,s2):
    m1 = m1.item()
    m2 = m2.item()
    std1 = std1.item()
    std2 = std2.item()
    s1 = s1.item()
    s2 = s2.item()
    a = 1/(2*std1**2) - 1/(2*std2**2)
    b = m2/(std2**2) - m1/(std1**2)
    c = m1**2 /(2*std1**2) - m2**2 / (2*std2**2) - np.log((std2*s1)/(std1*s2))
    return np.roots([a,b,c])

In [ ]:
def plot_scatter_no_ellipse(X, axes, row, col, title, colors, xlower=0, ylower=0, xupper=1, yupper=1):

    ## plot scatter
    axes[row][col].scatter(X[:, 0], X[:, 1], 0.8, c=colors, linewidth=0, marker='.')

    ## plot dividing lines
    for intercept in np.arange(0.2, 0.8, 0.1):
        axes[row][col].axline(xy1=(0.1, intercept), slope=-1, linestyle='--',
                              linewidth=0.3, color='tan')
    for intercept in np.arange(-0.9, 0.9, 0.1):
        axes[row][col].axline(xy1=(intercept,0.1), slope=1, linestyle='--', 
                              linewidth=0.3, color='tan')
        
    ## add immune gates
    axes[row][col].axline(xy1=(0, 0.7), slope=-1, linestyle='--', color='silver')
    axes[row][col].axline(xy1=(0, 0.6), slope=-1, linestyle='--', color='silver')
    axes[row][col].axline(xy1=(0, 0.5), slope=-1, linestyle='--', color='black')
    
    ## add epi gate
    #axes[row][col].axline(xy1=(0.3,0.1), slope=1, linestyle='--', color='red')
    axes[row][col].axline(xy1=(0.3,0), slope=1, linestyle='--', color='red')

    ## add mesenchymal gate 1
    axes[row][col].axline(xy1=(0,0.3), slope=1, linestyle='--', color='blue')
    
    axes[row][col].set_xlim([xlower, xupper])
    axes[row][col].set_ylim([ylower, yupper])
    axes[row][col].set_title(title)
    axes[row][col].tick_params()
    axes[row][col].tick_params()
    axes[row][col].grid(True, axis='both', linestyle='--', color='tan', linewidth=0.25)
    
    return axes

In [ ]:
'''
Rules for deciding clipping cutoffs:
1. For CD45 and VIM, we may use a similar procedure, because it is not assumed that these are the absolute majority populations.
2. In contrast, for EPCAM, there may be an absolute majority population (e.g. I3 samples).

3. For CD45/VIM, we may define the CLIP_LOW as the lowest_mean - 2*lowest_mean_SD.
4. Our task is to define CLIP_HIGH.
5. To do so, we calculate whether the difference between adjacent means is larger than the sum of the stdevs of the adjacent means.
6. If it is, then the means are well-separated; if not, then the means are not well-separated, and we consider the next pair of means.
7. If the middle_mean is well-separated from the lowest_mean, define CLIP_HIGH as middle_mean + 2*middle_mean_SD.
8. If not: if the highest_mean is well-separated from the middle_mean, define CLIP_HIGH as highest_mean.
9. If the highest_mean is still not well-separated from the middle_mean, define CLIP_HIGH as the highest_mean + 1*highest_mean_SD.

10. For EPCAM, our task is similar, except that, if all means are not well-separated, then CLIP_LOW is the 2nd percentile.
'''

In [ ]:
def plot_histogram_gmms(means, covs, weights, x, axes, order_to_use, rowind, colind=0):
    x_axis = np.arange(0, 4095, 1)
    y_axis0 = norm.pdf(x_axis, float(means[order_to_use[0]][0]), np.sqrt(float(covs[order_to_use[0]][0][0])))*weights[order_to_use[0]] # 1st gaussian
    y_axis1 = norm.pdf(x_axis, float(means[order_to_use[1]][0]), np.sqrt(float(covs[order_to_use[1]][0][0])))*weights[order_to_use[1]] # 2nd gaussian
    y_axis2 = norm.pdf(x_axis, float(means[order_to_use[2]][0]), np.sqrt(float(covs[order_to_use[2]][0][0])))*weights[order_to_use[2]] # 2nd gaussian

    axes[rowind].hist(x, density=True, color='black', bins=np.arange(0, 4095, 1))
    axes[rowind].plot(x_axis, y_axis0, lw=2, c='C0')
    axes[rowind].plot(x_axis, y_axis1, lw=2, c='C1')
    axes[rowind].plot(x_axis, y_axis2, lw=2, c='C2')
    axes[rowind].plot(x_axis, y_axis0 + y_axis1 + y_axis2, lw=2, c='C3', linestyle='--')

    # we add in the 1.4 * threshold lines
    axes[rowind].axvline(x= 1.4*float(means[order_to_use[0]][0]), linestyle='--', color='C0')
    axes[rowind].axvline(x= 1.4*float(means[order_to_use[1]][0]), linestyle='--', color='C1')
    
    axes[rowind].set_xlabel(f'Intensity value_ch{rowind}', fontsize=14)
    axes[rowind].set_ylabel(r"Density", fontsize=14)

    return axes

In [ ]:
def cluster_normalized(normalized, datasetid):
    inten_pd = pd.DataFrame(normalized, columns=['CD45', 'EPCAM', 'VIM'])
    inten_pd_to_return = inten_pd.copy()
    
    st = time.time()
    col_linkage = hierarchy.linkage(distance.pdist(inten_pd), method='ward')
    et = time.time()
    print(f'Calculating col_linkage took {et-st:.3f} seconds.')
    
    for ind, t_dist in enumerate([1,2,3,5,8,11,15]):
        hc = hierarchy.fcluster(col_linkage, t = t_dist, criterion = 'distance') #     

        ## add the results of the hc to the inten_pd dataframe
        inten_pd_to_return['DSC_t_dist_'+str(t_dist)] = hc 
        
        cm = plt.cm.get_cmap("tab20")

        # repeat the colormap, so that we don't run out of colors...
        original_colors = cm(np.linspace(0, 1, 20))
        n_repeats = int(np.ceil(max(hc)/20))
        repeated_colors = np.tile(original_colors, (n_repeats, 1))
        repeated_cm = ListedColormap(repeated_colors)
        
        cm_dict = dict(zip(range(max(hc)), [repeated_cm(i) for i in range(max(hc))]))
        print("Number of celltypes:", max(hc))
        cm_list = [cm_dict[lb-1] for lb in np.arange(1,max(hc)+1)]
        hcc = [cm_list[i-1] for i in hc]
        
        g = sns.clustermap(inten_pd.T, figsize = (8, 2), col_linkage = col_linkage, col_colors = hcc, \
                           cmap = 'viridis', colors_ratio=0.2, vmin=0.25, vmax=0.75)
        _ = g.ax_heatmap.set_xticklabels([])
        _ = g.ax_heatmap.set_xticks([])
        
        out_path = os.path.join(output_dir, f'{datasetid}_clustering_3_comp_GMM_dist_{t_dist}.png')
        plt.savefig(out_path)
        #plt.close(g)

    return inten_pd_to_return

In [ ]:
for i,dir in enumerate(dirs):
    datasetid = dir[:8]
    print(datasetid)
        
    os.chdir(os.path.join(master_dir, dir, 'Correct_peak_locations'))
    file = glob.glob('*_crypt_mem_wSigBRmv.csv')[0]
    data_raw = pd.read_csv(file)
    data = data_raw 
    
    inten_mean = data[['Raw_CD45_intensity_mean', 'Raw_EPCAM_intensity_mean', 'Raw_VIM_intensity_mean']].to_numpy()
    #print(inten_mean)
    #print(np.argwhere(np.isnan(inten_mean)))
    
    pre_normalized = np.zeros(inten_mean.shape)
    normalized = np.zeros(inten_mean.shape)

    clips = {'CD45':[], 'EPCAM':[], 'VIM':[]}
    
    for index in range(3):
        lower = np.percentile(inten_mean[:, index], 2)
        mid = np.percentile(inten_mean[:, index], 50)
        upper = np.percentile(inten_mean[:, index], 98)
        gmm = mixture.GaussianMixture(n_components=3, covariance_type="full", random_state=123, 
                                      means_init=([[lower], [mid], [upper]]))
        
        array = inten_mean[:, index]
        #print(array.shape)
        quantile_low = np.quantile(array, 0) #.0001) #0.001)
        quantile_high = np.quantile(array, 1) #.9999) #0.999)
        #print(str(quantile_low), ' / ', str(quantile_high))
        subset = array[(array >= quantile_low) & (array <= quantile_high)]
        print(subset.shape)
        
        gmm.fit(subset.reshape(-1, 1))
        #print(gmm.means_.shape)
        order_to_use = np.argsort(gmm.means_[:,0])

        #plot_histogram_gmms(gmm.means_, gmm.covariances_, gmm.weights_, array, axes_hist, order_to_use, index, 0)
        #plot_histogram_gmms(gmm.means_, gmm.covariances_, gmm.weights_, array, axes_hist2, order_to_use, index, 0)

        #print(f'{datasetid}: Channel {index}, 2nd:{int(np.quantile(array, 0.02))}, 90th:{int(np.quantile(array, 0.90))}, 95th:{int(np.quantile(array, 0.95))}, 98th:{int(np.quantile(array, 0.98))}, 1_0:{gmm.means_[order_to_use[1]]/gmm.means_[order_to_use[0]]}, 2_1:{gmm.means_[order_to_use[2]]/gmm.means_[order_to_use[1]]}')
        #print(f'{datasetid}: Channel {index}: {gmm.weights_[order_to_use[0]]}, {gmm.means_[order_to_use[0]]}, {np.sqrt(gmm.covariances_[order_to_use[0]])}')
        #print(f'{datasetid}: Channel {index}: {gmm.weights_[order_to_use[1]]}, {gmm.means_[order_to_use[1]]}, {np.sqrt(gmm.covariances_[order_to_use[1]])}')
        #print(f'{datasetid}: Channel {index}: {gmm.weights_[order_to_use[2]]}, {gmm.means_[order_to_use[2]]}, {np.sqrt(gmm.covariances_[order_to_use[2]])}')

        clip_low = 0; clip_high = 0
        # define clip_low:
        if index%2==0: # for CD45, VIM
            #print(f'Now looking at channel: {index}, with gmm and covariance: {gmm.means_[order_to_use[0]]}, {np.sqrt(gmm.covariances_[order_to_use[0]])}')
            clip_low = gmm.means_[order_to_use[0]] - 2*np.sqrt(gmm.covariances_[order_to_use[0]])
            #print(f'Setting clip_low to be: {clip_low}')
        else: # for EPCAM, determine if there is a mono-population:
            #print(f'For EPCAM: {(gmm.means_[order_to_use[0]] + np.sqrt(gmm.covariances_[order_to_use[0]]) + np.sqrt(gmm.covariances_[order_to_use[1]]))}\
            #    {gmm.means_[order_to_use[1]]}, {(gmm.means_[order_to_use[1]] + np.sqrt(gmm.covariances_[order_to_use[1]]) + np.sqrt(gmm.covariances_[order_to_use[2]]))}\
            #    {gmm.means_[order_to_use[2]]}')
            #print(f'{((gmm.means_[order_to_use[0]] + np.sqrt(gmm.covariances_[order_to_use[0]]) + np.sqrt(gmm.covariances_[order_to_use[1]])) < gmm.means_[order_to_use[1]])}, \
            #{((gmm.means_[order_to_use[1]] + np.sqrt(gmm.covariances_[order_to_use[1]]) + np.sqrt(gmm.covariances_[order_to_use[2]])) < gmm.means_[order_to_use[2]])}')
            if ((gmm.means_[order_to_use[0]] + np.sqrt(gmm.covariances_[order_to_use[0]]) + np.sqrt(gmm.covariances_[order_to_use[1]])) > gmm.means_[order_to_use[1]]) & \
               ((gmm.means_[order_to_use[1]] + np.sqrt(gmm.covariances_[order_to_use[1]]) + np.sqrt(gmm.covariances_[order_to_use[2]])) > gmm.means_[order_to_use[2]]):
                clip_low = np.quantile(array, 0.02)
            else:
                clip_low = gmm.means_[order_to_use[0]] - 2*np.sqrt(gmm.covariances_[order_to_use[0]])

        split_point_1 = solve(gmm.means_[order_to_use[0]], gmm.means_[order_to_use[1]], \
                            np.sqrt(gmm.covariances_[order_to_use[0]]), np.sqrt(gmm.covariances_[order_to_use[1]]), \
                            gmm.weights_[order_to_use[0]], gmm.weights_[order_to_use[1]]
                            )
        split_point_2 = solve(gmm.means_[order_to_use[1]], gmm.means_[order_to_use[2]], \
                            np.sqrt(gmm.covariances_[order_to_use[1]]), np.sqrt(gmm.covariances_[order_to_use[2]]), \
                            gmm.weights_[order_to_use[1]], gmm.weights_[order_to_use[2]]
                            )
        print(f'{datasetid}: Channel {index} split_points: {max(split_point_1)}, {max(split_point_2)}')
        # define clip_high:
        # we need to first separate EPCAM/VIM from CD45:
        if index%3==0: # for CD45 only
            # here we look to see whether the means are at all separated from each other, using ratio=1.4:
            # middle not separated, high separated: do largest mean
            # middle not separated, high not separated: do largest mean + 2*sd
            # middle separated, high separated: do split point between median and largest mean.
            # middle separated, high not separated: do median mean + 2*sd
            
            # if middle_mean is well-separated from lowest_mean
            if (gmm.means_[order_to_use[1]] / gmm.means_[order_to_use[0]]) > 1.4 :
                print(f'middle well separated from low')
                if (gmm.means_[order_to_use[2]] / gmm.means_[order_to_use[1]]) > 1.4 :
                    print(f'high also well separated from middle')
                    clip_high = max(split_point_2) 
                else:
                    print(f'high not well separated from middle')
                    clip_high = gmm.means_[order_to_use[1]] + 2*np.sqrt(gmm.covariances_[order_to_use[1]]) 
            else:
                print(f'middle not well separated from low')
                if (gmm.means_[order_to_use[2]] / gmm.means_[order_to_use[1]]) > 1.4 :
                    print(f'high well separated from middle')
                    clip_high = gmm.means_[order_to_use[2]] 
                else:
                    print(f'high also not well separated from middle')
                    clip_high = gmm.means_[order_to_use[2]] + 2*np.sqrt(gmm.covariances_[order_to_use[2]]) 
            clips['CD45'] = [clip_low, clip_high]
        elif index%3==1: # for EPCAM
            # here we look to see whether the means are at all separated from each other, using ratio=1.4:
            # middle not separated, high separated: do largest mean
            # middle not separated, high not separated: do largest mean + 2*sd
            # middle separated, high separated: do split point between median and largest mean.
            # middle separated, high not separated: do median mean + 2*sd
            
            # if middle_mean is well-separated from lowest_mean
            if (gmm.means_[order_to_use[1]] / gmm.means_[order_to_use[0]]) > 1.4 :
                print(f'middle well separated from low')
                if (gmm.means_[order_to_use[2]] / gmm.means_[order_to_use[1]]) > 1.4 :
                    print(f'high also well separated from middle')
                    clip_high =  max(split_point_2) 
                else:
                    print(f'high not well separated from middle')
                    clip_high = gmm.means_[order_to_use[1]] + 2*np.sqrt(gmm.covariances_[order_to_use[1]]) 
            else:
                print(f'middle not well separated from low')
                if (gmm.means_[order_to_use[2]] / gmm.means_[order_to_use[1]]) > 1.4 :
                    print(f'high well separated from middle')
                    clip_high = gmm.means_[order_to_use[2]] 
                else:
                    print(f'high also not well separated from middle')
                    clip_high = gmm.means_[order_to_use[2]] + 2*np.sqrt(gmm.covariances_[order_to_use[2]]) 
            clips['EPCAM'] = [clip_low, clip_high]
        else: # for  VIM
            # here we look to see whether the means are at all separated from each other, using ratio=1.4:
            # middle not separated, high separated: do largest mean
            # middle not separated, high not separated: do largest mean + 2*sd
            # middle separated, high separated: do split point between median and largest mean.
            # middle separated, high not separated: do median mean + 2*sd
            
            # if middle_mean is well-separated from lowest_mean
            if (gmm.means_[order_to_use[1]] / gmm.means_[order_to_use[0]]) > 1.4 :
                print(f'middle well separated from low')
                if (gmm.means_[order_to_use[2]] / gmm.means_[order_to_use[1]]) > 1.4 :
                    print(f'high also well separated from middle')
                    clip_high = max(split_point_2) 
                else:
                    print(f'high not well separated from middle')
                    clip_high = gmm.means_[order_to_use[1]] + 2*np.sqrt(gmm.covariances_[order_to_use[1]])
            else:
                print(f'middle not well separated from low')
                if (gmm.means_[order_to_use[2]] / gmm.means_[order_to_use[1]]) > 1.4 :
                    print(f'high well separated from middle')
                    clip_high = gmm.means_[order_to_use[2]] 
                else:
                    print(f'high also not well separated from middle')
                    clip_high = gmm.means_[order_to_use[2]] + 2*np.sqrt(gmm.covariances_[order_to_use[2]]) 
            clips['VIM'] = [clip_low, clip_high]
        print(f'{datasetid}: {index}: CLIP_LOW:{clip_low}, CLIP_HIGH:{clip_high}')        
        
        array_F = (array - clip_low)/(clip_high - clip_low)
        pre_normalized[:, index] = np.clip(array_F, 0, 1)

    ## first, save the dictionary of clips:
    clips_df = pd.DataFrame(clips)
    clips_df.to_csv(datasetid + '_DSC_clips.csv')
    
    sums = np.sum(pre_normalized, axis=1)
    print(f'Number of cells with pre_normalized sum==0: {np.sum(sums==0)}')
    sums[sums==0] = 1 # just in case the pre_normalized sum is 0.
    normalized[:,0] = pre_normalized[:,0] / sums #np.sum(pre_normalized, axis=1)
    normalized[:,1] = pre_normalized[:,1] / sums #np.sum(pre_normalized, axis=1)
    normalized[:,2] = pre_normalized[:,2] / sums #np.sum(pre_normalized, axis=1)

    EV_norm = normalized[:, 1:]

    ##
    ## do a classification, and then plot the scatterplot of epi and mesenchymal
    mesenchymal_filter = ((normalized[:, 1] + normalized[:, 2])>0.7) & ((normalized[:,2] - normalized[:,1])>0.3).astype(int)
    epi_filter = ((normalized[:, 1] + normalized[:, 2])>0.7) & ((normalized[:,1] - normalized[:,2])>0.3).astype(int)
    epi_final_filter = ((normalized[:, 1] + normalized[:, 2])>0.5) & ((normalized[:,1] - normalized[:,2])>0.3).astype(int)
    comb_filter = mesenchymal_filter + 2*epi_filter
    comb_final_filter = mesenchymal_filter + 2*epi_final_filter
    
    ct_dict = {1:'mesenchymal', 2:'epithelial', 0:'undefined'}
    comb = [ct_dict[key] for key in comb_filter]
    comb_final = [ct_dict[key] for key in comb_final_filter]
    data['new_ct_class'] = comb
    data['final_celltype'] = comb_final

    epi = data[['new_ct_class']].value_counts()['epithelial']/data.shape[0]
    mesenchymal = data[['new_ct_class']].value_counts()['mesenchymal']/data.shape[0]
    print(f'{datasetid}: EPCAM: {epi}, {epi*data.shape[0]}, VIM: {mesenchymal}, {mesenchymal*data.shape[0]}')

    epi_final = data[['final_celltype']].value_counts()['epithelial']/data.shape[0]
    mesenchymal_final = data[['final_celltype']].value_counts()['mesenchymal']/data.shape[0]
    print(f'{datasetid}: EPCAM: {epi_final}, {epi_final*data.shape[0]}, VIM: {mesenchymal_final}, {mesenchymal_final*data.shape[0]}')
     
    data.to_csv(file, index=False)